In [4]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel


#version transformers==4.53.3 worked

In [6]:
!pip install bitsandbytes==0.41.1 --default-timeout=600

  Using cached bitsandbytes-0.41.1-py3-none-any.whl.metadata (9.8 kB)
Using cached bitsandbytes-0.41.1-py3-none-any.whl (92.6 MB)


In [5]:
!pip uninstall -y bitsandbytes triton

Found existing installation: bitsandbytes 0.41.1
Uninstalling bitsandbytes-0.41.1:
  Successfully uninstalled bitsandbytes-0.41.1
Found existing installation: triton 3.7.0
Uninstalling triton-3.7.0:
  Successfully uninstalled triton-3.7.0


In [ ]:
!pip install transformers==4.53.3

In [5]:
#Per usare il modello fine-tuned per generare storie
# Percorsi
base_model_name = "microsoft/Phi-4-mini-instruct"
#base_model_name = "meta-llama/Meta-Llama-3-8B"
#base_model_name = "google/gemma-2-2b"
adapter_path = "phi4_ss_gen_final"
#adapter_path = "gemma2_ss_gen_final"
#adapter_path = "llama3_ss_gen_final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16 # P100 preferisce float16
)

# 1. Carica il modello base (lo stesso usato per il training)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 2. Carica il Tokenizer salvato
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Carica gli adapter (i pesi allenati da te) sopra il modello base
model = PeftModel.from_pretrained(base_model, adapter_path)

# Ora il 'model' è quello intelligente che sa scrivere Social Stories!
model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(200064, 3072, padding_idx=199999)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (qkv_proj): lora.Linear4bit(
                (base_la

In [6]:
ds = load_dataset("FMiMiY/SS-GEN", split='test')

In [7]:
#Zero-shot
def tokenizer_finetuned(batch):
    #template riportato dalla figura 5 del paper
    prompt = (
        "Develop a concise, clear, straightforward, positive and supportive "
        "Social Story titled \"{title}\" for children and teens with autism, "
        "200-300 words, that promotes their social understanding and boosts "
        "their participation in daily activities, fostering independence and confidence."
    )

    texts = [
        f"{prompt.format(title=t)}\nTitle: {t}\n\nSocial Story:"
        for t in batch['title']
    ]
    return {"text": texts}


In [8]:
tokenized_data = ds.map(tokenizer_finetuned,batched=True, remove_columns=ds.column_names)

Map:   0%|          | 0/508 [00:00<?, ? examples/s]

In [ ]:
import json
import os
import torch
from tqdm.auto import tqdm

# Percorso del file su Google Drive
checkpoint_path = "SS_GEN_results_phi4_4B_finetuned.json"

# 1. Carica i risultati precedenti se esistono (RESUME)
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        results = json.load(f)
    print(f"Rilevato checkpoint: riparto dalla storia numero {len(results)}")
else:
    results = []
    print("Nessun checkpoint trovato: inizio da capo.")

start_index = len(results)

# 2. Ciclo di generazione
for i in tqdm(range(start_index, len(tokenized_data)), desc="Generazione con Checkpoint"):

    # Prepara l'input
    inputs = tokenizer(tokenized_data[i]['text'], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

    # Aggiungi il nuovo risultato
    results.append({
        "id": i,
        "prompt": tokenized_data[i]['text'],
        "generated_story": generated_text.strip()
    })

    # 3. Salva su Drive ogni 5 iterazioni (CHECKPOINT)
    if i % 5 == 0:
        with open(checkpoint_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4, ensure_ascii=False)

# Salvataggio finale definitivo
with open(checkpoint_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"\nLavoro completato! Totale storie: {len(results)}")

Nessun checkpoint trovato: inizio da capo.


Generazione con Checkpoint:   0%|          | 0/508 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
import json

# Creiamo una lista di dizionari con tutti i dati
data_to_save = []
test_titles = ds['title']
for i in range(len(test_titles)):
    entry = {
        "id": i,
        "title": test_titles[i],
        "reference_story": ds['story_content'][i],
        "generated_story": results[i]  # La lista dei risultati del tuo loop
    }
    data_to_save.append(entry)

# Salvataggio su file
with open('risultati_finetuned_phi4.json', 'w', encoding='utf-8') as f:
    json.dump(data_to_save, f, ensure_ascii=False, indent=4)

print("Risultati salvati con successo!")